# Koeltekaart Amsterdam — Unified Analysis Pipeline

Computational social science pipeline for RQ1–RQ7.

---
## Section 0: Setup & Configuration

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json, os, sys, difflib, math, time, urllib.request, urllib.parse, glob, re
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.regression.quantile_regression import QuantReg
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
print(f'numpy {np.__version__}  pandas {pd.__version__}  matplotlib {matplotlib.__version__}')

try:
    import geopandas as gpd
    from shapely.geometry import Point, box
    HAS_GEO = True; print(f'geopandas {gpd.__version__} OK')
except ImportError:
    HAS_GEO = False; print('geopandas not available')

try:
    import libpysal; from libpysal.weights import Queen
    import esda; from esda.moran import Moran, Moran_Local
    HAS_ESDA = True; print(f'esda {esda.__version__} OK')
except ImportError:
    HAS_ESDA = False; print('esda/libpysal not available')

try:
    import osmnx as ox; import networkx as nx
    HAS_OSMNX = True; print(f'osmnx {ox.__version__} OK')
except ImportError:
    HAS_OSMNX = False; print('osmnx not available — pip install osmnx')

try:
    import ee; HAS_EE = True; print('earthengine-api OK')
except ImportError:
    HAS_EE = False; print('earthengine-api not available — pip install earthengine-api')

try:
    import pingouin as pg; HAS_PINGOUIN = True; print(f'pingouin {pg.__version__} OK')
except ImportError:
    HAS_PINGOUIN = False; print('pingouin not available')

try:
    import seaborn as sns; HAS_SNS = True
except ImportError:
    HAS_SNS = False; print('seaborn not available')

try:
    import folium; HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False

In [ ]:
AMS_BLUE='#004699'; AMS_HOVER='#003677'; AMS_RED='#ec0000'
AMS_GREEN='#00893c'; AMS_ORANGE='#ff9100'; AMS_CYAN='#009de6'
AMS_PURPLE='#a00078'; AMS_LIME='#bed200'; AMS_GREY='#767676'; AMS_TEXT='#202020'
TIER_PALETTE=[AMS_LIME,AMS_CYAN,AMS_ORANGE,AMS_RED,AMS_PURPLE]
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white',
    'axes.edgecolor':AMS_TEXT,'axes.labelcolor':AMS_TEXT,'text.color':AMS_TEXT,
    'xtick.color':AMS_TEXT,'ytick.color':AMS_TEXT,'font.family':'sans-serif','figure.dpi':100})

BASE     = Path('.')
CHARLIE  = BASE / 'charlie' / 'analysis'
DATA_RAW = BASE / '..' / 'data' / 'raw'
DATA_DIR = BASE / '..' / 'data'
OUTPUTS  = BASE / 'outputs'; OUTPUTS.mkdir(exist_ok=True)
AMF_GPK  = CHARLIE / 'ams_features.gpkg'
AMF_CSV  = CHARLIE / 'ams_features.csv'
CRD_CSV  = CHARLIE / 'cleaned_climate_risk_data.csv'
PC6_GPK  = DATA_RAW / '2025-cbs_pc6_2024_v1' / 'cbs_pc6_2024_v1.gpkg'
KOEL_CSV = DATA_DIR / 'koelteplekken.csv'
TREES_JSON = OUTPUTS / 'trees_raw.json'
WALK_GML   = OUTPUTS / 'amsterdam_walk.graphml'
SHADE_DIR  = BASE / '..' / 'frontend' / 'data' / 'shade'

print('Paths configured.')
for p, lbl in [(AMF_GPK,'ams_features.gpkg'),(AMF_CSV,'ams_features.csv'),
               (CRD_CSV,'climate_risk.csv'),(PC6_GPK,'cbs_pc6.gpkg'),(KOEL_CSV,'koelteplekken.csv')]:
    print(f'  {lbl:<30} exists: {p.exists()}')

---
## Section 1: Data Loading & Cleaning

In [ ]:
if HAS_GEO and AMF_GPK.exists():
    gdf = gpd.read_file(AMF_GPK)
    df  = pd.DataFrame(gdf.drop(columns='geometry'))
    print(f'Loaded ams_features.gpkg  shape={gdf.shape}  CRS={gdf.crs}')
elif AMF_CSV.exists():
    df  = pd.read_csv(AMF_CSV, low_memory=False)
    gdf = None
    print(f'Loaded ams_features.csv   shape={df.shape}  (no geometry)')
else:
    raise FileNotFoundError('Neither ams_features.gpkg nor ams_features.csv found.')
print(df[['buurtcode','buurtnaam','temp_mean','ndvi_mean','water_prc','road_prc']].head(3))

In [ ]:
CBS_SENTINELS = [-100000000, -99999999, -99999997, -99999996]
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for s in CBS_SENTINELS:
    df[num_cols] = df[num_cols].replace(s, np.nan)
df[num_cols] = df[num_cols].where(df[num_cols] >= -9e7, np.nan)
print(f'Total NaN cells after sentinel cleaning: {df.isnull().sum().sum()}')
print('Top 10 columns by missing count:')
print(df.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
crd = pd.read_csv(CRD_CSV, low_memory=False)
print(f'Climate risk data: {crd.shape}')

def strip_prefix(s):
    return str(s).replace('RK_','').strip() if pd.notna(s) else ''

crd['_name_clean'] = crd['name'].apply(strip_prefix)
df_names = df['buurtnaam'].dropna().tolist()

def fuzzy_match(name, choices, cutoff=0.6):
    m = difflib.get_close_matches(name, choices, n=1, cutoff=cutoff)
    return m[0] if m else None

crd['_buurtnaam_match'] = crd['_name_clean'].apply(
    lambda x: fuzzy_match(x, df_names) if x else None)
matched = crd['_buurtnaam_match'].notna().sum()
print(f'Fuzzy match: {matched}/{len(crd)} CRD rows matched ({100*matched/len(crd):.1f}%)')

HEAT_COLS = ['HI_TOTAAL_S.0','DR_AV_MATE_VAN_VERHARDING_V.0','DR_BS_GROEN_PUBLIEK_V.0',
             'DR_BS_GROEN_PRIVAAT_V.0','WO_GV_OPPERVLAKTE_WATER_V.0',
             'HI_BS_SCHADUW_OP_LOOPGEBIEDEN_V.0','HI_AV_TOEGANG_TOT_TUIN_V.0']
present_hc = [c for c in HEAT_COLS if c in crd.columns]
crd_merge = (crd[['_buurtnaam_match']+present_hc]
             .rename(columns={'_buurtnaam_match':'buurtnaam'})
             .dropna(subset=['buurtnaam']))
df = df.merge(crd_merge, on='buurtnaam', how='left', suffixes=('','_crd'))
print(f'After merge: {df["HI_TOTAAL_S.0"].notna().sum()}/{len(df)} buurten have heat-risk data')

In [ ]:
pc6_gdf = None
if HAS_GEO and PC6_GPK.exists():
    try:
        print('Loading CBS PC6 2024 (Amsterdam bbox)...')
        pc6_gdf = gpd.read_file(PC6_GPK, bbox=(4.7, 52.25, 5.1, 52.45))
        print(f'PC6 loaded: {pc6_gdf.shape}  CRS={pc6_gdf.crs}')
        print('PC6 columns:', pc6_gdf.columns.tolist()[:15])
    except Exception as e:
        print(f'Bbox load failed ({e}), trying rows=50000...')
        try:
            pc6_gdf = gpd.read_file(PC6_GPK, rows=50000)
            print(f'PC6 sampled: {pc6_gdf.shape}')
        except Exception as e2:
            print(f'PC6 load failed: {e2}'); pc6_gdf = None
elif not PC6_GPK.exists():
    print(f'CBS PC6 not found at {PC6_GPK}')
else:
    print('geopandas not available — cannot load PC6')
HAS_PC6 = pc6_gdf is not None and len(pc6_gdf) > 0
print(f'HAS_PC6 = {HAS_PC6}')

In [ ]:
print('=== DATA LOADING REPORT ===')
print(f'df shape: {df.shape}')
key_vars = ['temp_mean','ndvi_mean','water_prc','road_prc','HI_TOTAAL_S.0',
            'DR_AV_MATE_VAN_VERHARDING_V.0','DR_BS_GROEN_PUBLIEK_V.0','aantalInwoners']
for v in key_vars:
    if v in df.columns:
        n = int(df[v].isna().sum()); pct = 100*n/len(df)
        print(f'  {v:<50} missing={n} ({pct:.1f}%)')

---
## Section 2: Tree Data Acquisition & Buurt Aggregation

In [ ]:
WFS_BASE = ('https://api.data.amsterdam.nl/v1/wfs/bomen'
            '?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature'
            '&TYPENAMES=app:stamgegevens&OUTPUTFORMAT=application/json&COUNT=10000')

def fetch_trees(save_path):
    all_features = []
    offset = 0
    while True:
        url = f'{WFS_BASE}&STARTINDEX={offset}'
        print(f'  Fetching batch starting at offset {offset}...', end=' ', flush=True)
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                data = json.loads(r.read().decode('utf-8'))
        except Exception as e:
            print(f'ERROR: {e}'); break
        features = data.get('features', [])
        print(f'{len(features)} features')
        if not features: break
        all_features.extend(features)
        if len(features) < 10000: break
        offset += 10000; time.sleep(0.3)
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(all_features, f)
    print(f'  Saved {len(all_features):,} features to {save_path}')
    return all_features

if TREES_JSON.exists():
    print(f'Loading cached tree data from {TREES_JSON}...')
    with open(TREES_JSON,'r',encoding='utf-8') as f:
        tree_features = json.load(f)
    print(f'Loaded {len(tree_features):,} trees from cache.')
else:
    print('Fetching tree data from Amsterdam WFS...')
    tree_features = fetch_trees(TREES_JSON)
print(f'Total trees: {len(tree_features):,}')

In [ ]:
HEIGHT_MAP = {'a. tot 6 m.':3.0,'b. 6 tot 9 m.':7.5,'c. 9 tot 12 m.':10.5,
              'd. 12 tot 15 m.':13.5,'e. 15 tot 18 m.':16.5,'f. 18 tot 24 m.':21.0,
              'g. 24 m. en hoger':26.0}
MATURE = {'d. 12 tot 15 m.','e. 15 tot 18 m.','f. 18 tot 24 m.','g. 24 m. en hoger'}

records = []
for feat in tree_features:
    p = feat.get('properties', {})
    records.append({
        'buurt_id':   str(p.get('gbd_buurt_id','') or '').strip(),
        'height_cls': str(p.get('boomhoogteklasse_actueel','') or '').strip(),
        'species':    str(p.get('soortnaam','') or '').strip(),
    })
trees = pd.DataFrame(records)
trees = trees[trees['buurt_id'].str.len() > 0].copy()
trees['height_num'] = trees['height_cls'].map(HEIGHT_MAP)
trees['is_mature']  = trees['height_cls'].isin(MATURE)

sample_ids = trees['buurt_id'].dropna().unique()[:3].tolist()
sample_bc  = df['buurtcode'].dropna().astype(str).unique()[:3].tolist()
print('gbd_buurt_id sample:', sample_ids)
print('buurtcode sample:   ', sample_bc)
id_len = max((len(x) for x in sample_ids if x), default=14)
df['_bc_norm'] = df['buurtcode'].astype(str).str.zfill(id_len)
trees['_bid_norm'] = trees['buurt_id'].str.zfill(id_len)

agg = (trees.groupby('_bid_norm')
       .agg(tree_count=('buurt_id','count'),
            mature_count=('is_mature','sum'),
            mean_height_score=('height_num','mean'),
            tree_species_richness=('species','nunique'))
       .reset_index().rename(columns={'_bid_norm':'_bc_norm'}))
agg['pct_mature_trees'] = agg['mature_count'] / agg['tree_count'] * 100

df = df.merge(agg, on='_bc_norm', how='left')
if 'buurt_area' in df.columns:
    med_area = df['buurt_area'].dropna().median()
    df['_area_km2'] = df['buurt_area'] / (1e6 if med_area > 1e5 else 1)
    df['tree_density_per_km2'] = df['tree_count'] / df['_area_km2'].replace(0, np.nan)
else:
    df['tree_density_per_km2'] = np.nan

n_match = df['tree_count'].notna().sum()
print(f'Buurten matched: {n_match}/{len(df)} ({100*n_match/len(df):.1f}%)')
print(df[['buurtnaam','tree_count','tree_density_per_km2','pct_mature_trees']].dropna(subset=['tree_count']).head(5))

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,5))
td = df['tree_density_per_km2'].dropna()
axes[0].hist(td, bins=30, color=AMS_BLUE, edgecolor='white')
axes[0].axvline(td.median(), color=AMS_RED, linestyle='--', label=f'Median {td.median():.0f}')
axes[0].set_title('Tree Density Distribution', fontweight='bold')
axes[0].set_xlabel('Trees per km²'); axes[0].set_ylabel('Buurten'); axes[0].legend()

top10 = df.nlargest(10,'tree_density_per_km2')[['buurtnaam','tree_density_per_km2']]
axes[1].barh(top10['buurtnaam'], top10['tree_density_per_km2'], color=AMS_GREEN)
axes[1].set_title('Top 10 — Highest Tree Density', fontweight='bold')
axes[1].set_xlabel('Trees per km²'); axes[1].invert_yaxis()

bot10 = df[df['tree_count']>0].nsmallest(10,'tree_density_per_km2')[['buurtnaam','tree_density_per_km2']]
axes[2].barh(bot10['buurtnaam'], bot10['tree_density_per_km2'], color=AMS_ORANGE)
axes[2].set_title('Bottom 10 — Lowest Tree Density', fontweight='bold')
axes[2].set_xlabel('Trees per km²'); axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUTS/'tree_density.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 3: OSMnx Network Setup

In [ ]:
G_walk = None
if HAS_OSMNX:
    if WALK_GML.exists():
        print(f'Loading cached walk graph from {WALK_GML}...')
        G_walk = ox.load_graphml(WALK_GML)
        print(f'Graph: {len(G_walk.nodes):,} nodes, {len(G_walk.edges):,} edges')
    else:
        print('Fetching Amsterdam walk network from OSM...')
        try:
            G_walk = ox.graph_from_place('Amsterdam, Netherlands', network_type='walk')
            ox.save_graphml(G_walk, WALK_GML)
            print(f'Saved  nodes={len(G_walk.nodes):,}  edges={len(G_walk.edges):,}')
        except Exception as e:
            print(f'OSMnx fetch failed: {e}'); G_walk = None
else:
    print('osmnx not installed. Install: pip install osmnx')
    print('Network distance calculations will be skipped.')

In [ ]:
df['dist_nearest_spot_km'] = np.nan
df['osmnx_dist_nearest_km'] = np.nan

def haversine_km(lat1,lon1,lat2,lon2):
    R=6371.0; phi1,phi2=math.radians(lat1),math.radians(lat2)
    dphi=math.radians(lat2-lat1); dlam=math.radians(lon2-lon1)
    a=math.sin(dphi/2)**2+math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R*2*math.atan2(math.sqrt(a),math.sqrt(1-a))

if KOEL_CSV.exists():
    koel = pd.read_csv(KOEL_CSV).dropna(subset=['latitude','longitude'])
    print(f'Koelteplekken: {len(koel)} spots')
    kc = koel[['latitude','longitude']].values
    if HAS_GEO and gdf is not None:
        cents = gdf.to_crs('EPSG:4326').geometry.centroid
        df['_clat'] = cents.y.values; df['_clon'] = cents.x.values
    elif 'latitude' in df.columns:
        df['_clat'] = df['latitude']; df['_clon'] = df['longitude']
    else:
        df['_clat'] = np.nan; df['_clon'] = np.nan

    dists = []
    for _, row in df.iterrows():
        if pd.isna(row['_clat']):
            dists.append(np.nan); continue
        d = sorted(haversine_km(row['_clat'],row['_clon'],k[0],k[1]) for k in kc)
        dists.append(d[0] if d else np.nan)
    df['dist_nearest_spot_km'] = dists
    print(f'Straight-line distances: mean={np.nanmean(dists):.2f} km')
else:
    print(f'koelteplekken.csv not found at {KOEL_CSV}')

if HAS_OSMNX and G_walk is not None and KOEL_CSV.exists() and '_clat' in df.columns:
    print('Computing OSMnx distances for top-20 HVI buurten...')
    proxy = df['HI_TOTAAL_S.0'].fillna(df['temp_mean']).fillna(0)
    for idx in proxy.nlargest(20).index:
        row = df.loc[idx]
        if pd.isna(row['_clat']): continue
        try:
            orig = ox.nearest_nodes(G_walk, row['_clon'], row['_clat'])
            best = np.inf
            for k in kc[:50]:
                dest = ox.nearest_nodes(G_walk, k[1], k[0])
                try:
                    l = nx.shortest_path_length(G_walk, orig, dest, weight='length')
                    best = min(best, l)
                except nx.NetworkXNoPath: pass
            df.at[idx,'osmnx_dist_nearest_km'] = best/1000 if best < np.inf else np.nan
        except Exception: pass
    print(f'Network distances: {df["osmnx_dist_nearest_km"].notna().sum()} buurten')
print(df[['buurtnaam','dist_nearest_spot_km','osmnx_dist_nearest_km']].dropna(subset=['dist_nearest_spot_km']).head(5))

---
## Section 4: LST — Daytime + Nighttime

In [ ]:
print('temp_mean = daytime Landsat LST from GEE (already in ams_features)')
print(f'Range: {df["temp_mean"].min():.2f}–{df["temp_mean"].max():.2f} °C')
print(f'Mean: {df["temp_mean"].mean():.2f} °C  Std: {df["temp_mean"].std():.2f} °C')

fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].hist(df['temp_mean'].dropna(), bins=30, color=AMS_BLUE, edgecolor='white')
axes[0].axvline(df['temp_mean'].mean(), color=AMS_RED, linestyle='--',
                label=f'Mean {df["temp_mean"].mean():.1f}°C')
axes[0].set_title('Daytime LST Distribution', fontweight='bold')
axes[0].set_xlabel('Surface Temperature (°C)'); axes[0].set_ylabel('Buurten')
axes[0].legend()
if HAS_GEO and gdf is not None:
    gdf_p = gdf.copy(); gdf_p['temp_mean'] = df['temp_mean'].values
    gdf_p.plot(column='temp_mean', ax=axes[1], cmap='RdYlBu_r', legend=True,
               missing_kwds={'color':'lightgrey'})
    axes[1].set_title('Daytime LST by Buurt', fontweight='bold'); axes[1].axis('off')
else:
    axes[1].scatter(range(len(df)), df['temp_mean'].sort_values().values, s=8, color=AMS_BLUE, alpha=0.6)
    axes[1].set_title('Daytime LST Sorted', fontweight='bold')
    axes[1].set_xlabel('Rank'); axes[1].set_ylabel('°C')
plt.tight_layout()
plt.savefig(OUTPUTS/'lst_daytime.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
temp_night_gee = False
if HAS_EE:
    try:
        ee.Authenticate(); ee.Initialize(project='your-gee-project-id')
        ams_geom = ee.Geometry.Rectangle([4.7, 52.25, 5.1, 52.45])
        modis_night = (ee.ImageCollection('MODIS/061/MOD11A2')
                       .filterDate('2023-06-01','2023-09-30')
                       .filterBounds(ams_geom).select('LST_Night_1km')
                       .mean().multiply(0.02).subtract(273.15).clip(ams_geom))
        if HAS_GEO and gdf is not None:
            gdf_wgs = gdf.to_crs('EPSG:4326')
            temps = []
            for _, row in gdf_wgs.iterrows():
                try:
                    feat = ee.Feature(ee.Geometry(row.geometry.__geo_interface__))
                    val = modis_night.reduceRegion(ee.Reducer.mean(),feat.geometry(),1000).getInfo()
                    temps.append(val.get('LST_Night_1km', np.nan))
                except Exception: temps.append(np.nan)
            df['temp_night'] = temps; temp_night_gee = True
            print(f'GEE nighttime LST: {df["temp_night"].notna().sum()} buurten')
    except Exception as e:
        print(f'GEE failed: {e}')
else:
    print('earthengine-api not installed.')
if not temp_night_gee:
    df['temp_night'] = df['temp_mean'] * 0.82 + 4.3
    print('NOTE: temp_night is a PROXY (temp_mean * 0.82 + 4.3) — empirical day-night correction')
print(f'temp_night: mean={df["temp_night"].mean():.2f}°C  std={df["temp_night"].std():.2f}°C')

---
## Section 5: EDA & Distribution Analysis

In [ ]:
PHYS_VARS = [v for v in ['temp_mean','ndvi_mean','water_prc','road_prc',
    'HI_TOTAAL_S.0','DR_AV_MATE_VAN_VERHARDING_V.0','DR_BS_GROEN_PUBLIEK_V.0',
    'DR_BS_GROEN_PRIVAAT_V.0','WO_GV_OPPERVLAKTE_WATER_V.0',
    'HI_BS_SCHADUW_OP_LOOPGEBIEDEN_V.0','tree_density_per_km2','pct_mature_trees']
    if v in df.columns]
SOCIAL_VARS = [v for v in ['percentagePersonen65JaarEnOuder',
    'percentageEenpersoonshuishoudens','aantalWmoClientenPer1000Inwoners',
    'percentagePersonenMetLaagInkomen','percentageHuishoudensOnderOfRondSociaalMinimum',
    'percentageNietWesterseMigratieachtergrond','bevolkingsdichtheidInwonersPerKm2']
    if v in df.columns]
DIST_VARS = [v for v in ['bibliotheekGemiddeldeAfstandInKm','zwembadGemiddeldeAfstandInKm',
    'treinstationGemiddeldeAfstandInKm','huisartsenpraktijkGemiddeldeAfstandInKm',
    'dist_nearest_spot_km'] if v in df.columns]
ALL_KEY = PHYS_VARS + SOCIAL_VARS + DIST_VARS
print(f'Physical ({len(PHYS_VARS)}): {PHYS_VARS}')
print(f'Social   ({len(SOCIAL_VARS)}): {SOCIAL_VARS}')
print(f'Distance ({len(DIST_VARS)}): {DIST_VARS}')

In [ ]:
desc = df[ALL_KEY].describe().T
desc['cv']   = desc['std'] / desc['mean'].abs()
desc['skew'] = df[ALL_KEY].skew()
desc['kurt'] = df[ALL_KEY].kurt()
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', None)
print(desc[['count','mean','std','min','max','cv','skew','kurt']].to_string())

In [ ]:
def plot_hist_grid(vlist, title, color):
    n = len(vlist)
    if n == 0: print(f'No vars for: {title}'); return
    ncols = 4; nrows = math.ceil(n/ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4*nrows))
    axes = np.array(axes).flatten()
    for i, var in enumerate(vlist):
        data = df[var].dropna()
        axes[i].hist(data, bins=25, color=color, edgecolor='white', density=True, alpha=0.75)
        if len(data) > 3:
            kx = np.linspace(data.min(), data.max(), 200)
            axes[i].plot(kx, stats.gaussian_kde(data)(kx), color=AMS_RED, lw=2)
        axes[i].set_title(var[:30], fontsize=9, fontweight='bold')
        axes[i].tick_params(labelsize=8)
    for j in range(i+1, len(axes)): axes[j].axis('off')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUTS/f'hist_{title[:15].replace(" ","_")}.png', dpi=100, bbox_inches='tight')
    plt.show()

plot_hist_grid(PHYS_VARS,   'Physical Variables',  AMS_BLUE)
plot_hist_grid(SOCIAL_VARS, 'Social Vulnerability', AMS_PURPLE)
plot_hist_grid(DIST_VARS,   'Distance Variables',  AMS_CYAN)

In [ ]:
pvars = [v for v in ALL_KEY if df[v].notna().sum() >= 10]
std_d = df[pvars].apply(lambda x: (x-x.mean())/x.std())
fig, ax = plt.subplots(figsize=(18,6))
bp = ax.boxplot([std_d[v].dropna().values for v in pvars], patch_artist=True)
for p in bp['boxes']: p.set_facecolor(AMS_BLUE); p.set_alpha(0.7)
ax.set_xticks(range(1,len(pvars)+1))
ax.set_xticklabels([v[:20] for v in pvars], rotation=45, ha='right', fontsize=8)
ax.axhline(0, color=AMS_RED, linestyle='--', alpha=0.5)
ax.set_title('Standardised Box Plot', fontweight='bold'); ax.set_ylabel('z-score')
plt.tight_layout()
plt.savefig(OUTPUTS/'boxplot_standardised.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
miss = df[ALL_KEY].isnull().astype(int)
fig, ax = plt.subplots(figsize=(14, max(4,len(ALL_KEY)//2)))
if HAS_SNS:
    sns.heatmap(miss.T, cmap=['white',AMS_RED], cbar=False,
                yticklabels=[v[:30] for v in ALL_KEY], ax=ax)
else:
    ax.imshow(miss.T.values, aspect='auto', cmap='Reds', interpolation='nearest')
    ax.set_yticks(range(len(ALL_KEY))); ax.set_yticklabels([v[:30] for v in ALL_KEY], fontsize=8)
ax.set_title('Missing Data Heatmap (red=missing)', fontweight='bold')
ax.set_xlabel('Buurt index')
plt.tight_layout()
plt.savefig(OUTPUTS/'missing_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 6: Normality Testing

In [ ]:
norm_rows = []
for var in ALL_KEY:
    data = df[var].dropna().values
    n = len(data)
    if n < 8: continue
    sw_s, sw_p = (stats.shapiro(data) if n<=5000 else (np.nan,np.nan))
    dk_s, dk_p = stats.normaltest(data)
    jb_s, jb_p = stats.jarque_bera(data)[:2]
    is_n = all(p>0.05 for p in [sw_p,dk_p,jb_p] if not np.isnan(p))
    norm_rows.append({'variable':var,'n':n,
        'SW_p':round(float(sw_p),4) if not np.isnan(sw_p) else np.nan,
        'DK_p':round(float(dk_p),4),'JB_p':round(float(jb_p),4),
        'Normal_a05':'YES' if is_n else 'NO'})
norm_df = pd.DataFrame(norm_rows)
print('Normality Tests:')
print(norm_df.to_string(index=False))

In [ ]:
qq_vars = (PHYS_VARS+SOCIAL_VARS)[:10]
fig, axes = plt.subplots(2, 5, figsize=(18,8))
axes = axes.flatten()
for i, var in enumerate(qq_vars):
    data = df[var].dropna().values
    if len(data) < 8: axes[i].axis('off'); continue
    (osm, osr), (slope, intercept, _) = stats.probplot(data)
    axes[i].scatter(osm, osr, color=AMS_BLUE, s=8, alpha=0.6)
    lx = np.array([min(osm),max(osm)])
    axes[i].plot(lx, slope*lx+intercept, color=AMS_RED, lw=1.5)
    axes[i].set_title(var[:22], fontsize=8, fontweight='bold')
    axes[i].set_xlabel('Theoretical Q', fontsize=7); axes[i].set_ylabel('Sample Q', fontsize=7)
for j in range(i+1,10): axes[j].axis('off')
fig.suptitle('Q-Q Plots (Normal)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUTS/'qq_plots.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 7: Outlier Detection

In [ ]:
out_flags = pd.DataFrame(index=df.index)
cap_log = []
for var in ALL_KEY:
    data = df[var].dropna()
    if len(data) < 10: continue
    Q1,Q3 = data.quantile(0.25), data.quantile(0.75); IQR=Q3-Q1
    iqr_flag = (df[var]<Q1-3*IQR)|(df[var]>Q3+3*IQR)
    z_flag   = np.abs(stats.zscore(df[var].fillna(data.mean()))) > 3.5
    out_flags[f'{var}_iqr'] = iqr_flag
    out_flags[f'{var}_z']   = z_flag
    n_iqr = int(iqr_flag.sum()); n_z = int(z_flag.sum())
    if n_iqr > 0:
        cap_log.append({'var':var,'n_iqr':n_iqr,'n_z':n_z})
        df[var] = df[var].clip(Q1-3*IQR, Q3+3*IQR)
df['n_outlier_vars'] = out_flags.sum(axis=1)
print('Capping decisions:')
if cap_log:
    for c in cap_log: print(f'  {c["var"][:45]} IQR_extreme={c["n_iqr"]} Z_extreme={c["n_z"]} → capped')
else:
    print('  No extreme outliers found.')
print('\nTop 10 buurten by outlier flag count:')
print(df.nlargest(10,'n_outlier_vars')[['buurtnaam','n_outlier_vars']].to_string(index=False))

In [ ]:
t20_idx = df.nlargest(20,'n_outlier_vars').index
hm_vars = [v for v in ALL_KEY if df[v].notna().sum()>=20][:15]
z_mat = df.loc[t20_idx, hm_vars].apply(lambda c: (c-c.mean())/c.std())
z_mat.index = df.loc[t20_idx,'buurtnaam'].values
fig, ax = plt.subplots(figsize=(16,8))
if HAS_SNS:
    sns.heatmap(z_mat, cmap='RdBu_r', center=0, annot=True, fmt='.1f',
                linewidths=0.4, ax=ax, annot_kws={'size':7})
else:
    im = ax.imshow(z_mat.values, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
    ax.set_xticks(range(len(hm_vars))); ax.set_xticklabels([v[:12] for v in hm_vars], rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(z_mat))); ax.set_yticklabels(z_mat.index, fontsize=8); plt.colorbar(im, ax=ax)
ax.set_title('Z-score Heatmap — Top 20 Outlier Buurten', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUTS/'outlier_zscore_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 8: Correlation & Covariance

In [ ]:
def corr_pvals(data):
    cols = data.columns; n = len(cols)
    pmat = pd.DataFrame(np.ones((n,n)), index=cols, columns=cols)
    for i,c1 in enumerate(cols):
        for j,c2 in enumerate(cols):
            if i!=j:
                m = data[[c1,c2]].dropna()
                if len(m)>4: _,p = stats.pearsonr(m[c1],m[c2]); pmat.loc[c1,c2]=p
    return pmat

def sig_stars(p):
    if p<0.001: return '***'
    if p<0.01:  return '**'
    if p<0.05:  return '*'
    return ''

cv = [v for v in ALL_KEY if df[v].notna().sum()>=20]
cd = df[cv].copy()
pr = cd.corr(method='pearson'); sr = cd.corr(method='spearman')
pv = corr_pvals(cd)
if 'HI_TOTAAL_S.0' in pr.columns:
    tc = pr['HI_TOTAAL_S.0'].drop('HI_TOTAAL_S.0').sort_values(key=abs, ascending=False)
    print('Top Pearson correlations with HI_TOTAAL_S.0:')
    for var,r in tc.head(10).items():
        p = pv.loc[var,'HI_TOTAAL_S.0'] if var in pv.index else np.nan
        print(f'  {var:<50} r={r:.3f} {sig_stars(p)}')

In [ ]:
pvc = cv[:14]
fig, axes = plt.subplots(1,2,figsize=(22,9))
for ax, cmat, title in zip(axes,[pr.loc[pvc,pvc],sr.loc[pvc,pvc]],['Pearson','Spearman']):
    if HAS_SNS:
        mask = np.triu(np.ones_like(cmat, dtype=bool))
        sns.heatmap(cmat, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                    annot=True, fmt='.2f', linewidths=0.4, ax=ax, annot_kws={'size':7},
                    xticklabels=[v[:18] for v in pvc], yticklabels=[v[:18] for v in pvc])
    else:
        im = ax.imshow(cmat.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
        ax.set_xticks(range(len(pvc))); ax.set_xticklabels([v[:15] for v in pvc], rotation=45, ha='right', fontsize=8)
        ax.set_yticks(range(len(pvc))); ax.set_yticklabels([v[:15] for v in pvc], fontsize=8)
        plt.colorbar(im, ax=ax)
    ax.set_title(f'{title} Correlation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUTS/'correlation_heatmaps.png', dpi=120, bbox_inches='tight')
plt.show()

if 'HI_TOTAAL_S.0' in pr.columns and 'DR_AV_MATE_VAN_VERHARDING_V.0' in pr.columns:
    r_imp = pr.loc['DR_AV_MATE_VAN_VERHARDING_V.0','HI_TOTAAL_S.0']
    print(f'Impervious surface: Pearson r={r_imp:.3f} with heat risk')
if 'DR_BS_GROEN_PUBLIEK_V.0' in pr.columns and 'HI_TOTAAL_S.0' in pr.columns:
    r_gr = pr.loc['DR_BS_GROEN_PUBLIEK_V.0','HI_TOTAAL_S.0']
    print(f'Public green: Pearson r={r_gr:.3f} with heat risk')

In [ ]:
sv_p = [v for v in SOCIAL_VARS if v in df.columns and df[v].notna().sum()>=20]
if len(sv_p)>=2:
    ktab = pd.DataFrame(index=sv_p, columns=sv_p, dtype=float)
    for v1 in sv_p:
        for v2 in sv_p:
            m = df[[v1,v2]].dropna()
            if len(m)>4: tau,_ = stats.kendalltau(m[v1],m[v2]); ktab.loc[v1,v2]=round(tau,3)
    print('Kendall τ — SVI variables:'); print(ktab.to_string())
else:
    print('Insufficient SVI variables for Kendall τ.')
cv_mat = cd[pvc[:8]].cov(); cv_norm = cv_mat/cv_mat.values.max()
print('\nNormalised covariance matrix (first 8 vars):'); print(cv_norm.round(3).to_string())

---
## Section 9: Multicollinearity (VIF)

In [ ]:
def compute_vif(predictors, data):
    pres = [p for p in predictors if p in data.columns]
    Xd = data[pres].dropna()
    if len(Xd) < len(pres)+2: print('Insufficient data for VIF.'); return pd.DataFrame()
    Xc = sm.add_constant(Xd)
    return pd.DataFrame({'Variable':Xc.columns,
        'VIF':[round(variance_inflation_factor(Xc.values,i),2) for i in range(Xc.shape[1])]})

reg_preds = [v for v in ['ndvi_mean','water_prc','road_prc',
    'DR_AV_MATE_VAN_VERHARDING_V.0','DR_BS_GROEN_PUBLIEK_V.0',
    'DR_BS_GROEN_PRIVAAT_V.0','WO_GV_OPPERVLAKTE_WATER_V.0','tree_density_per_km2']
    if v in df.columns]

vif_df = compute_vif(reg_preds, df)
print('VIF — Regression Predictors:')
for _, row in vif_df.iterrows():
    flag = ' ← HIGH' if row['VIF']>10 else (' ← MODERATE' if row['VIF']>5 else '')
    print(f'  {row["Variable"]:<50} VIF={row["VIF"]:.2f}{flag}')

high_vif = [v for v in vif_df[vif_df['VIF']>10]['Variable'].tolist() if v!='const']
if high_vif:
    print(f'\nSuggested drops (VIF>10): {high_vif}')
    red_preds = [p for p in reg_preds if p not in high_vif]
    vif2 = compute_vif(red_preds, df)
    print('VIF after drops:'); print(vif2.to_string(index=False))
else:
    print('\nNo VIF>10 — no drops needed.')

---
## Section 10: Social Vulnerability Index (PCA)

In [ ]:
SVI_VARS = [v for v in ['percentagePersonen65JaarEnOuder',
    'percentageEenpersoonshuishoudens','aantalWmoClientenPer1000Inwoners',
    'percentagePersonenMetLaagInkomen','percentageHuishoudensOnderOfRondSociaalMinimum',
    'percentageNietWesterseMigratieachtergrond'] if v in df.columns]
print(f'SVI variables: {SVI_VARS}')
svi_data = df[SVI_VARS].dropna()
print(f'Complete rows: {len(svi_data)}/{len(df)}')

def kmo_manual(X):
    corr = np.corrcoef(X.T); np.fill_diagonal(corr,0)
    inv  = np.linalg.pinv(np.corrcoef(X.T)); pv = np.zeros_like(corr)
    for i in range(len(corr)):
        for j in range(len(corr)):
            if i!=j: pv[i,j] = -inv[i,j]/math.sqrt(max(inv[i,i]*inv[j,j],1e-300))
    s_corr = np.sum(corr**2); s_part = np.sum(pv**2)
    return s_corr/(s_corr+s_part)

if len(SVI_VARS)>=3 and len(svi_data)>10:
    kmo = kmo_manual(svi_data.values)
    print(f'KMO={kmo:.3f} ({"Meritorious" if kmo>0.7 else "Mediocre" if kmo>0.5 else "Poor"})')
    corm = svi_data.corr().values; n_obs=len(svi_data)
    det = np.linalg.det(corm)
    chi2 = -(n_obs-1-(2*len(SVI_VARS)+5)/6)*math.log(max(det,1e-300))
    df_b = len(SVI_VARS)*(len(SVI_VARS)-1)//2
    p_b  = 1-stats.chi2.cdf(chi2,df_b)
    print(f'Bartlett χ²({df_b})={chi2:.2f} p={p_b:.4f} → {"Suitable" if p_b<0.05 else "NOT suitable"}')

In [ ]:
scaler = StandardScaler(); X_svi = scaler.fit_transform(svi_data)
pca = PCA(); pca.fit(X_svi)
fig, axes = plt.subplots(1,3,figsize=(18,5))

# Scree
nc = len(pca.explained_variance_ratio_)
axes[0].bar(range(1,nc+1), pca.explained_variance_ratio_*100, color=AMS_BLUE)
axes[0].plot(range(1,nc+1), np.cumsum(pca.explained_variance_ratio_)*100,
             color=AMS_RED, marker='o', lw=2, label='Cumulative')
axes[0].axhline(80, linestyle='--', color=AMS_GREY, label='80%')
axes[0].set_title('Scree Plot — SVI PCA', fontweight='bold')
axes[0].set_xlabel('PC'); axes[0].set_ylabel('Variance (%)'); axes[0].legend()

# Loading heatmap
loadings = pd.DataFrame(pca.components_[:min(3,nc)].T,
    index=[v[:22] for v in SVI_VARS], columns=[f'PC{i+1}' for i in range(min(3,nc))])
if HAS_SNS:
    sns.heatmap(loadings, cmap='RdBu_r', center=0, annot=True, fmt='.2f', linewidths=0.4, ax=axes[1])
else:
    im=axes[1].imshow(loadings.values,cmap='RdBu_r',aspect='auto',vmin=-1,vmax=1)
    axes[1].set_xticks(range(loadings.shape[1])); axes[1].set_xticklabels(loadings.columns)
    axes[1].set_yticks(range(len(SVI_VARS))); axes[1].set_yticklabels([v[:18] for v in SVI_VARS], fontsize=8)
    plt.colorbar(im,ax=axes[1])
axes[1].set_title('PCA Loadings', fontweight='bold')

# Biplot
scores = pca.transform(X_svi)
axes[2].scatter(scores[:,0],scores[:,1],color=AMS_BLUE,s=8,alpha=0.5)
for i,var in enumerate(SVI_VARS):
    axes[2].annotate(var[:15],xy=(pca.components_[0,i]*3,pca.components_[1,i]*3),
                     color=AMS_RED,fontsize=7,ha='center')
    axes[2].arrow(0,0,pca.components_[0,i]*3,pca.components_[1,i]*3,
                  head_width=0.05,color=AMS_RED,alpha=0.7)
axes[2].set_title('Biplot PC1 vs PC2', fontweight='bold')
axes[2].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[2].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.tight_layout()
plt.savefig(OUTPUTS/'svi_pca.png', dpi=120, bbox_inches='tight')
plt.show()

pc1 = scores[:,0]
eld = 'percentagePersonen65JaarEnOuder'
if eld in SVI_VARS and pca.components_[0,SVI_VARS.index(eld)]<0:
    pc1=-pc1; print('PC1 flipped: higher=more vulnerable')
pc1_norm = (pc1-pc1.min())/(pc1.max()-pc1.min())
df['svi_pca'] = np.nan; df.loc[svi_data.index,'svi_pca'] = pc1_norm
print(f'svi_pca: {df["svi_pca"].notna().sum()} buurten, range [{df["svi_pca"].min():.3f},{df["svi_pca"].max():.3f}]')

---
## Section 11: Multi-year Vulnerability Trends (RQ7)

In [ ]:
cbs_csvs = []
for yr in range(2018,2026):
    cbs_csvs += list(DATA_RAW.glob(f'*{yr}*buurt*.csv'))+list(DATA_RAW.glob(f'*{yr}*buurten*.csv'))
cbs_csvs += list(DATA_RAW.glob('cbs_*buurt*.csv'))+list(DATA_RAW.glob('*cbs*buurt*.csv'))
cbs_csvs = [f for f in set(cbs_csvs) if f.exists() and f.suffix=='.csv']
df['svi_trend_slope'] = np.nan; df['svi_trend_class'] = 'unknown'

if cbs_csvs:
    print(f'Found {len(cbs_csvs)} multi-year CBS files:')
    for f in cbs_csvs: print(f'  {f}')
    year_svis = {}
    for fpath in cbs_csvs:
        try:
            tmp = pd.read_csv(fpath, low_memory=False)
            for s in CBS_SENTINELS:
                tmp[tmp.select_dtypes(include=[np.number]).columns] = tmp.select_dtypes(include=[np.number]).replace(s,np.nan)
            ym = re.search(r'(20\d\d)', fpath.name)
            yr = int(ym.group(1)) if ym else None
            pres = [v for v in SVI_VARS if v in tmp.columns]
            if pres and yr:
                sv = tmp[pres].dropna()
                if len(sv)>10:
                    Xs = StandardScaler().fit_transform(sv)
                    pc = PCA(n_components=1).fit_transform(Xs)[:,0]
                    pcn = (pc-pc.min())/(pc.max()-pc.min())
                    year_svis[yr] = dict(zip(sv.index, pcn))
                    print(f'  {yr}: {len(sv)} buurten')
        except Exception as e: print(f'  {fpath.name}: {e}')
    if len(year_svis)>=2:
        yrs = sorted(year_svis.keys())
        bidxs = set.intersection(*[set(v.keys()) for v in year_svis.values()])
        slopes = {}
        for bi in bidxs:
            ys = np.array([year_svis[y].get(bi,np.nan) for y in yrs])
            xs = np.array(yrs,dtype=float); mask = ~np.isnan(ys)
            if mask.sum()>=3:
                slope,*_ = stats.linregress(xs[mask],ys[mask])
                slopes[bi] = slope
        sls = pd.Series(slopes, name='svi_trend_slope')
        df['svi_trend_slope'] = sls.reindex(df.index)
        df['svi_trend_class'] = df['svi_trend_slope'].apply(
            lambda s: 'worsening' if s>0.02 else ('improving' if s<-0.02 else 'stable') if pd.notna(s) else 'unknown')
        print('\nTrend classes:'); print(df['svi_trend_class'].value_counts())
        if 'HI_TOTAAL_S.0' in df.columns:
            lh = df['HI_TOTAAL_S.0']<df['HI_TOTAAL_S.0'].quantile(0.33)
            hot = df[lh&(df['svi_trend_class']=='worsening')]
            print(f'Emerging hotspots: {len(hot)} buurten')
            if len(hot)>0: print(hot[['buurtnaam','svi_trend_slope']].head(10).to_string(index=False))
        sv = df['svi_trend_slope'].dropna()
        fig,ax=plt.subplots(figsize=(10,5))
        ax.hist(sv,bins=30,color=AMS_BLUE,edgecolor='white')
        ax.axvline(0,color=AMS_RED,linestyle='--',label='No change')
        ax.set_title('SVI Trend Slopes',fontweight='bold'); ax.set_xlabel('Slope/yr'); ax.set_ylabel('Buurten'); ax.legend()
        plt.tight_layout(); plt.savefig(OUTPUTS/'svi_trend.png',dpi=120,bbox_inches='tight'); plt.show()
    else: print('Fewer than 2 years — trend analysis skipped.')
else:
    print('No multi-year CBS files found.')
    print(f'Deposit year-labelled CSVs in: {DATA_RAW}')
    print('Expected pattern: *<YYYY>*buurt*.csv')

---
## Section 12: RQ3 — Physical Environmental Drivers of Heat Risk

In [ ]:
target = 'HI_TOTAAL_S.0'
phys = [v for v in ['DR_AV_MATE_VAN_VERHARDING_V.0','DR_BS_GROEN_PUBLIEK_V.0',
    'DR_BS_GROEN_PRIVAAT_V.0','WO_GV_OPPERVLAKTE_WATER_V.0',
    'HI_BS_SCHADUW_OP_LOOPGEBIEDEN_V.0','tree_density_per_km2']
    if v in df.columns and target in df.columns]

if phys and target in df.columns:
    nc = min(len(phys),3); nr = math.ceil(len(phys)/nc)
    fig, axes = plt.subplots(nr, nc, figsize=(6*nc, 5*nr))
    axes = np.array(axes).flatten()
    for i,var in enumerate(phys):
        sub = df[[var,target]].dropna()
        axes[i].scatter(sub[var], sub[target], color=AMS_BLUE, s=10, alpha=0.4)
        if len(sub)>4:
            m,b,r,p,_ = stats.linregress(sub[var],sub[target])
            xr = np.linspace(sub[var].min(),sub[var].max(),100)
            axes[i].plot(xr, m*xr+b, color=AMS_RED, lw=2, label=f'r={r:.2f} p={p:.3f}')
        axes[i].set_xlabel(var[:25]); axes[i].set_ylabel('Heat Risk'); axes[i].legend(fontsize=8)
        axes[i].set_title(var[:28], fontweight='bold', fontsize=9)
    for j in range(i+1, len(axes)): axes[j].axis('off')
    plt.suptitle('Physical Drivers vs Heat Risk (RQ3)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUTS/'rq3_scatter.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Correlation bar chart
    corrs = [(v, df[[v,target]].dropna().corr().loc[v,target]) for v in phys]
    corrs.sort(key=lambda x: x[1])
    fig2, ax2 = plt.subplots(figsize=(10,5))
    colors2 = [AMS_GREEN if c<0 else AMS_RED for _,c in corrs]
    ax2.barh([v[:30] for v,_ in corrs], [c for _,c in corrs], color=colors2)
    ax2.axvline(0, color=AMS_TEXT, lw=0.8)
    ax2.set_title('Correlation with Heat Risk (RQ3)', fontweight='bold')
    ax2.set_xlabel('Pearson r')
    plt.tight_layout()
    plt.savefig(OUTPUTS/'rq3_corr_bar.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print(f'Target {target} not available — skipping RQ3 scatter.')

In [ ]:
def run_ols_hc3(y_col, x_cols, data, label='Model'):
    sub = data[[y_col]+x_cols].dropna()
    y = sub[y_col]; X = sm.add_constant(sub[x_cols])
    m = sm.OLS(y, X).fit(cov_type='HC3')
    print(f'\n{label}  R²={m.rsquared:.4f}  Adj-R²={m.rsquared_adj:.4f}  n={len(sub)}')
    print(m.summary2().tables[1].to_string())
    return m

base_x = [v for v in ['DR_AV_MATE_VAN_VERHARDING_V.0','DR_BS_GROEN_PUBLIEK_V.0',
    'DR_BS_GROEN_PRIVAAT_V.0','WO_GV_OPPERVLAKTE_WATER_V.0'] if v in df.columns]
ext_x  = base_x + (['tree_density_per_km2'] if 'tree_density_per_km2' in df.columns else [])

m_a = m_b = None
if base_x and 'HI_TOTAAL_S.0' in df.columns:
    m_a = run_ols_hc3('HI_TOTAAL_S.0', base_x, df, 'Model A (physical only)')
if len(ext_x)>len(base_x):
    m_b = run_ols_hc3('HI_TOTAAL_S.0', ext_x, df, 'Model B (+ tree density)')
    r2_imp = (m_b.rsquared - m_a.rsquared) if m_a else 0
    r2_imp_pct = r2_imp*100
    r2_a_pct   = m_a.rsquared*100 if m_a else 0
    print(f'\nImpervious surface explains {r2_a_pct:.1f}% of heat variance;')
    print(f'Adding tree density improves R² by {r2_imp_pct:.1f}%')

In [ ]:
def std_betas(model, data, x_cols, y_col):
    sub = data[[y_col]+x_cols].dropna()
    sy = sub[y_col].std()
    betas = {}
    for c in x_cols:
        if c in model.params and sub[c].std()>0:
            betas[c] = model.params[c] * sub[c].std() / sy
    return betas

if m_a is not None:
    betas_a = std_betas(m_a, df, base_x, 'HI_TOTAAL_S.0')
    fig, ax = plt.subplots(figsize=(9,4))
    items = sorted(betas_a.items(), key=lambda x: x[1])
    colors_b = [AMS_GREEN if v<0 else AMS_BLUE for _,v in items]
    ax.barh([k[:25] for k,_ in items], [v for _,v in items], color=colors_b)
    ax.axvline(0, color=AMS_TEXT, lw=0.8)
    ax.set_title('Standardised β — Model A (RQ3)', fontweight='bold')
    ax.set_xlabel('Standardised β')
    plt.tight_layout()
    plt.savefig(OUTPUTS/'rq3_std_beta.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
if m_a is not None:
    fig, axes = plt.subplots(2,2,figsize=(12,10))
    fitted = m_a.fittedvalues; resid = m_a.resid
    axes[0,0].scatter(fitted, resid, s=8, color=AMS_BLUE, alpha=0.5)
    axes[0,0].axhline(0, color=AMS_RED, lw=1); axes[0,0].set_title('Fitted vs Residuals')
    axes[0,0].set_xlabel('Fitted'); axes[0,0].set_ylabel('Residuals')
    (osm,osr),(sl,ic,_) = stats.probplot(resid)
    axes[0,1].scatter(osm,osr,s=8,color=AMS_BLUE,alpha=0.6)
    lx=np.array([min(osm),max(osm)]); axes[0,1].plot(lx,sl*lx+ic,color=AMS_RED,lw=1.5)
    axes[0,1].set_title('Q-Q Plot Residuals')
    axes[1,0].scatter(fitted,np.sqrt(np.abs(resid)),s=8,color=AMS_BLUE,alpha=0.5)
    axes[1,0].set_title('Scale-Location'); axes[1,0].set_xlabel('Fitted'); axes[1,0].set_ylabel('√|Residuals|')
    infl = m_a.get_influence(); lev = infl.hat_matrix_diag
    axes[1,1].scatter(lev, resid, s=8, color=AMS_BLUE, alpha=0.5)
    axes[1,1].set_title('Leverage vs Residuals'); axes[1,1].set_xlabel('Leverage'); axes[1,1].set_ylabel('Residuals')
    plt.suptitle('Residual Diagnostics — Model A', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUTS/'rq3_residuals.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## Section 13: RQ4 — Intervention Priority Scoring

In [ ]:
def minmax(s): r=s.max()-s.min(); return (s-s.min())/r if r>0 else s*0+0.5

pr_vars = {'heat':('HI_TOTAAL_S.0',True),'imper':('DR_AV_MATE_VAN_VERHARDING_V.0',True),
           'green':('DR_BS_GROEN_PUBLIEK_V.0',False),'tree':('tree_density_per_km2',False)}
for k,(col,up) in pr_vars.items():
    if col in df.columns and df[col].notna().sum()>10:
        df[f'{k}_norm'] = minmax(df[col].fillna(df[col].median()))
    else:
        df[f'{k}_norm'] = 0.5

df['intervention_priority'] = (
    df['heat_norm'] + df['imper_norm'] - df['green_norm'] -
    (df['tree_norm'] * 0.5 if 'tree_norm' in df.columns else 0))

df['priority_class'] = pd.qcut(df['intervention_priority'], q=5,
    labels=['Very Low','Low','Medium','High','Very High'])

top20 = df[['buurtnaam','HI_TOTAAL_S.0','DR_AV_MATE_VAN_VERHARDING_V.0',
            'DR_BS_GROEN_PUBLIEK_V.0','tree_density_per_km2',
            'intervention_priority','priority_class']].nlargest(20,'intervention_priority')
print('Top 20 Intervention Priority Buurten:')
print(top20.to_string(index=False))

In [ ]:
if HAS_GEO and gdf is not None:
    gdf_pr = gdf.copy(); gdf_pr['priority'] = df['intervention_priority'].values
    fig, ax = plt.subplots(figsize=(12,10))
    gdf_pr.plot(column='priority', ax=ax, cmap='RdYlGn_r', legend=True,
                missing_kwds={'color':'lightgrey'})
    ax.set_title('Intervention Priority Score by Buurt (RQ4)', fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUTS/'rq4_priority_map.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No geometry — skipping choropleth.')

out_cols = [c for c in ['buurtnaam','HI_TOTAAL_S.0','DR_AV_MATE_VAN_VERHARDING_V.0',
    'DR_BS_GROEN_PUBLIEK_V.0','intervention_priority','priority_class'] if c in df.columns]
df[out_cols].sort_values('intervention_priority',ascending=False).to_csv(
    OUTPUTS/'intervention_priority.csv', index=False)
print('Exported intervention_priority.csv')

---
## Section 14: RQ1 — Heat Stratification (Full Social Model)

In [ ]:
phys_base = [v for v in ['ndvi_mean','water_prc','road_prc','buurt_area'] if v in df.columns]
phys_tree = phys_base + (['tree_density_per_km2'] if 'tree_density_per_km2' in df.columns else [])
soc_vars  = [v for v in ['percentagePersonen65JaarEnOuder','percentagePersonenMetLaagInkomen',
    'percentageNietWesterseMigratieachtergrond'] if v in df.columns]
phys_soc  = phys_tree + soc_vars

models14 = {}
r2_table  = []
for label, x_cols in [('A_physical',phys_base),('B_phys+tree',phys_tree),('C_full',phys_soc)]:
    x_pres = [v for v in x_cols if v in df.columns]
    if not x_pres: continue
    sub = df[['temp_mean']+x_pres].dropna()
    if len(sub)<20: continue
    y = sub['temp_mean']; X = sm.add_constant(sub[x_pres])
    m = sm.OLS(y,X).fit(cov_type='HC3')
    models14[label]=m; r2_table.append({'Model':label,'R2':round(m.rsquared,4),'Adj_R2':round(m.rsquared_adj,4),'n':len(sub)})
    print(f'\nModel {label}  R²={m.rsquared:.4f}  n={len(sub)}')
    print(m.summary2().tables[1].to_string())

print('\nR² Comparison:')
print(pd.DataFrame(r2_table).to_string(index=False))

In [ ]:
if 'C_full' in models14 and 'A_physical' in models14:
    r2_phys = models14['A_physical'].rsquared
    r2_full = models14['C_full'].rsquared
    partial_social = r2_full - r2_phys
    print(f'Partial R² for social block: {partial_social:.4f} ({partial_social*100:.1f}%)')
    print(f'Physical block explains {r2_phys*100:.1f}%, social adds {partial_social*100:.1f}%')

if 'C_full' in models14:
    m = models14['C_full']
    sub = df[['temp_mean']+phys_soc].dropna()
    sy = sub['temp_mean'].std()
    betas = {c: m.params[c]*sub[c].std()/sy for c in phys_soc if c in m.params and sub[c].std()>0}
    items = sorted(betas.items(),key=lambda x:x[1])
    fig,ax=plt.subplots(figsize=(10,5))
    ax.barh([k[:30] for k,_ in items],[v for _,v in items],
            color=[AMS_GREEN if v<0 else AMS_BLUE for _,v in items])
    ax.axvline(0,color=AMS_TEXT,lw=0.8)
    ax.set_title('Standardised β — Model C (RQ1)', fontweight='bold'); ax.set_xlabel('Standardised β')
    plt.tight_layout(); plt.savefig(OUTPUTS/'rq1_std_beta.png',dpi=120,bbox_inches='tight'); plt.show()

In [ ]:
if HAS_ESDA and HAS_GEO and gdf is not None and 'A_physical' in models14:
    try:
        from libpysal.weights import Queen as QW
        gdf_m = gdf.copy()
        sub = df[['temp_mean']+phys_base].dropna()
        gdf_sub = gdf_m.loc[sub.index]
        W = QW.from_dataframe(gdf_sub, silence_warnings=True)
        W.transform = 'r'
        resid_a = models14['A_physical'].resid
        mi = Moran(resid_a.values, W, permutations=999)
        print(f'Moran\'s I on Model A residuals: I={mi.I:.4f}  p={mi.p_sim:.4f}')
        if mi.p_sim < 0.05:
            print('Spatial autocorrelation detected — spatial lag model recommended.')
        else:
            print('No significant spatial autocorrelation in residuals.')
    except Exception as e:
        print(f'Moran\'s I failed: {e}')
else:
    print('esda/geopandas not available — Moran\'s I skipped.')

In [ ]:
if 'C_full' in models14 and HAS_GEO and gdf is not None:
    m = models14['C_full']
    sub = df[['temp_mean']+phys_soc].dropna()
    gdf_res = gdf.copy()
    gdf_res['residual'] = np.nan
    gdf_res.loc[sub.index, 'residual'] = m.resid.values
    fig, ax = plt.subplots(figsize=(12,10))
    gdf_res.plot(column='residual', ax=ax, cmap='RdBu_r', legend=True,
                 missing_kwds={'color':'lightgrey'}, vmin=-2, vmax=2)
    ax.set_title('Residuals Model C (UHI outliers in red)', fontweight='bold')
    ax.axis('off')
    plt.tight_layout(); plt.savefig(OUTPUTS/'rq1_residual_map.png',dpi=120,bbox_inches='tight'); plt.show()
    outliers = gdf_res[gdf_res['residual']>gdf_res['residual'].quantile(0.9)].copy()
    outliers['buurtnaam'] = df.loc[outliers.index,'buurtnaam'].values
    print('Urban heat island outliers (top 10% positive residuals):')
    print(outliers[['buurtnaam','residual']].sort_values('residual',ascending=False).head(10).to_string(index=False))

---
## Section 15: Spatial Autocorrelation (LISA)

In [ ]:
if not (HAS_ESDA and HAS_GEO and gdf is not None):
    print('esda/libpysal/geopandas not available — spatial autocorrelation skipped.')
    print('Install: pip install esda libpysal geopandas')
else:
    try:
        from libpysal.weights import Queen as QW
        gdf_l = gdf.copy()
        lisa_vars = [(v,'hvi') for v in ['HI_TOTAAL_S.0'] if v in df.columns]
        lisa_vars += [('temp_mean','temp'),('svi_pca','svi')]
        lisa_vars = [(v,lbl) for v,lbl in lisa_vars if v in df.columns and df[v].notna().sum()>20]

        W = QW.from_dataframe(gdf_l, silence_warnings=True); W.transform='r'
        print(f'Queen weights: {len(W.neighbors)} observations')

        for var, lbl in lisa_vars:
            vals = df[var].fillna(df[var].median()).values
            mi   = Moran(vals, W, permutations=999)
            ml   = Moran_Local(vals, W, permutations=999)
            print(f'\n{var}: Global Moran I={mi.I:.4f} p={mi.p_sim:.4f}')
            if mi.p_sim<0.05:
                print(f'  Significant spatial autocorrelation (p<0.05)')
            quad_labels = {1:'HH',2:'LH',3:'LL',4:'HL'}
            sig = ml.p_sim < 0.05
            gdf_l[f'lisa_{lbl}'] = 'ns'
            for q,ql in quad_labels.items():
                mask = (ml.q==q)&sig
                gdf_l.loc[mask, f'lisa_{lbl}'] = ql
            counts = gdf_l[f'lisa_{lbl}'].value_counts()
            print(f'  Cluster counts: {counts.to_dict()}')
            hh_pct = 100*counts.get('HH',0)/sig.sum() if sig.sum()>0 else 0
            print(f'  {hh_pct:.1f}% of significant buurten are HH clusters')

        if lisa_vars:
            var0, lbl0 = lisa_vars[0]
            cmap_lisa = {'HH':AMS_RED,'LH':AMS_CYAN,'LL':AMS_BLUE,'HL':AMS_ORANGE,'ns':'lightgrey'}
            fig, ax = plt.subplots(figsize=(12,10))
            for cat, color in cmap_lisa.items():
                sub = gdf_l[gdf_l[f'lisa_{lbl0}']==cat]
                if len(sub): sub.plot(ax=ax, color=color, label=cat)
            handles = [mpatches.Patch(color=c,label=l) for l,c in cmap_lisa.items()]
            ax.legend(handles=handles, title='LISA cluster', loc='lower right')
            ax.set_title(f'LISA Clusters — {var0}', fontweight='bold'); ax.axis('off')
            plt.tight_layout(); plt.savefig(OUTPUTS/'lisa_map.png',dpi=120,bbox_inches='tight'); plt.show()
    except Exception as e:
        print(f'LISA failed: {e}')

---
## Section 16: RQ2 — Cooling Access & Double Disadvantage

In [ ]:
access_dist_vars = [v for v in ['bibliotheekGemiddeldeAfstandInKm',
    'zwembadGemiddeldeAfstandInKm','treinstationGemiddeldeAfstandInKm',
    'huisartsenpraktijkGemiddeldeAfstandInKm'] if v in df.columns]

if access_dist_vars:
    for v in access_dist_vars:
        mn,mx = df[v].min(),df[v].max(); rng=mx-mn
        df[f'_inv_{v}'] = 1 - ((df[v]-mn)/rng) if rng>0 else 0.5
    inv_cols = [f'_inv_{v}' for v in access_dist_vars]
    df['cooling_access'] = df[inv_cols].mean(axis=1)
    if 'dist_nearest_spot_km' in df.columns:
        mn,mx = df['dist_nearest_spot_km'].min(),df['dist_nearest_spot_km'].max()
        inv_spot = 1 - ((df['dist_nearest_spot_km']-mn)/(mx-mn)) if (mx-mn)>0 else 0.5
        df['cooling_access'] = 0.6*df['cooling_access'] + 0.4*inv_spot
    df['cooling_access'] = (df['cooling_access']-df['cooling_access'].min()) / (df['cooling_access'].max()-df['cooling_access'].min()+1e-9)
    print(f'cooling_access: mean={df["cooling_access"].mean():.3f}  std={df["cooling_access"].std():.3f}')
else:
    df['cooling_access'] = np.nan
    print('No distance variables available for cooling access score.')

In [ ]:
if 'svi_pca' in df.columns and 'cooling_access' in df.columns:
    sub = df[['svi_pca','cooling_access','buurtnaam']].dropna()
    svi_med = sub['svi_pca'].median(); ca_med = sub['cooling_access'].median()
    fig, ax = plt.subplots(figsize=(10,8))
    ax.scatter(sub['svi_pca'], sub['cooling_access'], color=AMS_BLUE, s=15, alpha=0.5)
    ax.axvline(svi_med, color=AMS_GREY, linestyle='--', lw=1)
    ax.axhline(ca_med,  color=AMS_GREY, linestyle='--', lw=1)
    for q, (xs,xe,ys,ye), label, col in [
        (1,(svi_med,sub['svi_pca'].max(),ca_med,sub['cooling_access'].max()),'High SVI / High Access',AMS_LIME),
        (2,(sub['svi_pca'].min(),svi_med,ca_med,sub['cooling_access'].max()),'Low SVI / High Access',AMS_CYAN),
        (3,(sub['svi_pca'].min(),svi_med,sub['cooling_access'].min(),ca_med),'Low SVI / Low Access',AMS_ORANGE),
        (4,(svi_med,sub['svi_pca'].max(),sub['cooling_access'].min(),ca_med),'High SVI / Low Access ← double disadvantage',AMS_RED)
    ]:
        n = ((sub['svi_pca']>=xs)&(sub['svi_pca']<xe)&(sub['cooling_access']>=ys)&(sub['cooling_access']<ye)).sum()
        xm=(xs+xe)/2; ym=(ys+ye)/2
        ax.annotate(f'{label}\nn={n}', xy=(xm,ym), ha='center', fontsize=8, color=col, fontweight='bold')
    ax.set_xlabel('SVI (higher=more vulnerable)'); ax.set_ylabel('Cooling Access (higher=better)')
    ax.set_title('SVI vs Cooling Access — Double Disadvantage Analysis (RQ2)', fontweight='bold')
    r,p = stats.pearsonr(sub['svi_pca'],sub['cooling_access'])
    rs,ps = stats.spearmanr(sub['svi_pca'],sub['cooling_access'])
    ax.set_title(f'SVI vs Cooling Access (Pearson r={r:.3f} p={p:.3f}, Spearman ρ={rs:.3f})', fontweight='bold')
    plt.tight_layout(); plt.savefig(OUTPUTS/'rq2_double_disadvantage.png',dpi=120,bbox_inches='tight'); plt.show()
    print(f'Pearson r={r:.3f} p={p:.4f} — {"significant" if p<0.05 else "not significant"} at α=0.05')
    print(f'Spearman ρ={rs:.3f} p={ps:.4f}')
else:
    print('svi_pca or cooling_access not available — double disadvantage plot skipped.')

In [ ]:
if 'svi_pca' in df.columns and 'cooling_access' in df.columns:
    sub = df[['svi_pca','cooling_access']].dropna()
    y = sub['cooling_access'].values; X = sm.add_constant(sub['svi_pca'].values)
    taus = [0.10,0.25,0.50,0.75,0.90]
    coefs = []
    for tau in taus:
        qr = QuantReg(y,X).fit(q=tau)
        coefs.append({'tau':tau,'coef':qr.params[1],'ci_lo':qr.conf_int()[1,0],'ci_hi':qr.conf_int()[1,1]})
    qdf = pd.DataFrame(coefs)
    fig,ax=plt.subplots(figsize=(8,5))
    ax.plot(qdf['tau'],qdf['coef'],color=AMS_BLUE,marker='o',lw=2,label='SVI coeff')
    ax.fill_between(qdf['tau'],qdf['ci_lo'],qdf['ci_hi'],alpha=0.2,color=AMS_BLUE)
    ax.axhline(0,color=AMS_RED,linestyle='--',lw=1)
    ax.set_title('Quantile Regression: SVI → Cooling Access', fontweight='bold')
    ax.set_xlabel('Quantile τ'); ax.set_ylabel('Coefficient')
    plt.tight_layout(); plt.savefig(OUTPUTS/'rq2_quantreg.png',dpi=120,bbox_inches='tight'); plt.show()
    print(qdf.to_string(index=False))
else:
    print('Quantile regression skipped — missing svi_pca or cooling_access.')

In [ ]:
if 'svi_pca' in df.columns and 'cooling_access' in df.columns:
    sub = df[['svi_pca','cooling_access']].dropna().sort_values('svi_pca')
    cum_svi = np.arange(1,len(sub)+1)/len(sub)
    cum_access = sub['cooling_access'].cumsum()/sub['cooling_access'].sum()
    fig,ax=plt.subplots(figsize=(8,6))
    ax.plot(cum_svi, cum_access, color=AMS_BLUE, lw=2, label='Concentration curve')
    ax.plot([0,1],[0,1], color=AMS_GREY, linestyle='--', lw=1, label='Perfect equality')
    ax.fill_between(cum_svi, cum_access, cum_svi,
                    where=(cum_access<cum_svi), alpha=0.2, color=AMS_RED,
                    label='Inequality area (vulnerable have less access)')
    ax.set_title('Concentration Curve — Cooling Access by SVI rank', fontweight='bold')
    ax.set_xlabel('Cumulative population rank (low → high SVI)')
    ax.set_ylabel('Cumulative cooling access')
    ax.legend(); plt.tight_layout()
    plt.savefig(OUTPUTS/'rq2_concentration_curve.png',dpi=120,bbox_inches='tight')
    plt.show()

---
## Section 17: Composite HVI

In [ ]:
def norm01(s):
    rng = s.max()-s.min()
    return (s-s.min())/rng if rng>0 else s*0+0.5

if 'HI_TOTAAL_S.0' in df.columns:
    df['hi_norm'] = norm01(df['HI_TOTAAL_S.0'].fillna(df['HI_TOTAAL_S.0'].median()))
else:
    df['hi_norm'] = norm01(df['temp_mean'].fillna(df['temp_mean'].median()))

if 'svi_pca' in df.columns:
    svi_c = df['svi_pca'].fillna(df['svi_pca'].median())
else:
    svi_c = pd.Series(0.5, index=df.index)

if 'cooling_access' in df.columns:
    ca_c = df['cooling_access'].fillna(df['cooling_access'].median())
else:
    ca_c = pd.Series(0.5, index=df.index)

df['hvi'] = 0.40*df['hi_norm'] + 0.40*svi_c + 0.20*(1-ca_c)
df['hvi_tier'] = pd.qcut(df['hvi'], q=5, labels=['Tier1_Low','Tier2','Tier3','Tier4','Tier5_High'])

# Validate
if 'HI_TOTAAL_S.0' in df.columns:
    sub = df[['hvi','HI_TOTAAL_S.0']].dropna()
    r,p = stats.pearsonr(sub['hvi'],sub['HI_TOTAAL_S.0'])
    print(f'HVI vs HI_TOTAAL_S.0: Pearson r={r:.3f} p={p:.4f}')
    print(f'Expected strong positive correlation — {"OK" if r>0.5 else "WEAK — check inputs"}')

top20_hvi = df.nlargest(20,'hvi')[['buurtnaam','hvi','hi_norm','svi_pca','cooling_access','hvi_tier']]
print('\nTop 20 HVI Buurten:')
print(top20_hvi.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,6))
axes[0].hist(df['hvi'].dropna(), bins=30, color=AMS_BLUE, edgecolor='white')
axes[0].set_title('HVI Distribution', fontweight='bold')
axes[0].set_xlabel('HVI (0=low,1=high)'); axes[0].set_ylabel('Buurten')
for t,c in zip(['Tier1_Low','Tier2','Tier3','Tier4','Tier5_High'],TIER_PALETTE):
    v = df[df['hvi_tier']==t]['hvi'].quantile(0.5) if len(df[df['hvi_tier']==t])>0 else np.nan
    if not np.isnan(v): axes[0].axvline(v, color=c, linestyle='--', lw=1, label=t)
axes[0].legend(fontsize=8)
if 'HI_TOTAAL_S.0' in df.columns:
    sub = df[['hvi','HI_TOTAAL_S.0']].dropna()
    axes[1].scatter(sub['HI_TOTAAL_S.0'],sub['hvi'],s=8,color=AMS_BLUE,alpha=0.5)
    axes[1].set_xlabel('HI_TOTAAL_S.0 (klimaatrisicokaarten)'); axes[1].set_ylabel('HVI composite')
    axes[1].set_title('HVI vs Official Heat Risk Score', fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUTS/'hvi_overview.png',dpi=120,bbox_inches='tight'); plt.show()

---
## Section 18: Health Proxy (HSEI)

In [ ]:
def norm01s(s): rng=s.max()-s.min(); return (s-s.min())/rng if rng>0 else s*0+0.5

wmo_col = 'aantalWmoClientenPer1000Inwoners'
eld_col = 'percentagePersonen65JaarEnOuder'
dens_col= 'bevolkingsdichtheidInwonersPerKm2'

hsei_inputs = ['temp_mean', eld_col, dens_col, wmo_col]
avail = [v for v in hsei_inputs if v in df.columns]
print(f'HSEI inputs available: {avail}')

if len(avail)==4:
    df['hsei'] = (norm01s(df['temp_mean'].fillna(df['temp_mean'].median())) *
                  norm01s(df[eld_col].fillna(df[eld_col].median())/100) *
                  norm01s(df[dens_col].fillna(df[dens_col].median())) *
                  norm01s(df[wmo_col].fillna(df[wmo_col].median())))
    df['hsei'] = norm01s(df['hsei'])
    print(f'HSEI: mean={df["hsei"].mean():.3f}  std={df["hsei"].std():.3f}')

    if 'hvi' in df.columns:
        sub = df[['hsei','hvi']].dropna()
        r,p = stats.pearsonr(sub['hsei'],sub['hvi'])
        print(f'Cross-validation: HSEI vs HVI Pearson r={r:.3f} p={p:.4f}')
        print(f'{"Positive correlation confirmed — HSEI validates HVI" if r>0.3 and p<0.05 else "Weak/non-significant — review inputs"}')

    top10_hsei = df.nlargest(10,'hsei')[['buurtnaam','hsei']].reset_index(drop=True)
    top10_hvi  = df.nlargest(10,'hvi')[['buurtnaam','hvi']].reset_index(drop=True) if 'hvi' in df.columns else pd.DataFrame()
    print('\nTop 10 by HSEI:'); print(top10_hsei.to_string(index=False))
    if len(top10_hvi)>0:
        overlap = set(top10_hsei['buurtnaam'])&set(top10_hvi['buurtnaam'])
        print(f'\nTop 10 HVI:'); print(top10_hvi.to_string(index=False))
        print(f'Overlap top-10 HSEI vs top-10 HVI: {len(overlap)}/10 buurten — {overlap}')
elif len(avail)<4:
    df['hsei'] = np.nan
    print(f'HSEI requires temp_mean, {eld_col}, {dens_col}, {wmo_col}. Missing: {set(hsei_inputs)-set(avail)}')

---
## Section 19: RQ5 — Green Infrastructure Compensation

In [ ]:
df['served'] = (df['dist_nearest_spot_km'] <= 1.0).astype(float) if 'dist_nearest_spot_km' in df.columns else np.nan
n_served   = int(df['served'].sum()) if 'served' in df.columns else 0
n_unserved = int((df['served']==0).sum()) if 'served' in df.columns else 0
print(f'Served (≤1km koelteplek): {n_served}  Unserved: {n_unserved}')

green_vars = [v for v in ['ndvi_mean','tree_density_per_km2','water_prc','road_prc'] if v in df.columns]

if 'served' in df.columns and green_vars:
    unserved = df[df['served']==0].copy()
    y_u = unserved['temp_mean'].dropna()
    x_cols_u = [v for v in green_vars if unserved[v].notna().sum()>=10]
    if len(x_cols_u)>0 and len(y_u)>10:
        sub_u = unserved[['temp_mean']+x_cols_u].dropna()
        Xu = sm.add_constant(sub_u[x_cols_u])
        m_u = sm.OLS(sub_u['temp_mean'],Xu).fit(cov_type='HC3')
        print(f'\nUnserved buurten OLS  R²={m_u.rsquared:.4f}  n={len(sub_u)}')
        print(m_u.summary2().tables[1].to_string())
        if 'ndvi_mean' in m_u.params:
            ndvi_coef = m_u.params['ndvi_mean']
            ndvi_se   = m_u.bse['ndvi_mean']
            print(f'\nNDVI cooling: {ndvi_coef*0.1:.3f}°C per 0.1 NDVI unit (SE={ndvi_se*0.1:.3f})')

if 'served' in df.columns and green_vars:
    sub_full = df[['temp_mean','served']+green_vars].dropna()
    Xf = sub_full[green_vars+['served']].copy()
    if 'ndvi_mean' in Xf.columns:
        Xf['ndvi_x_served'] = Xf['ndvi_mean']*Xf['served']
    if 'tree_density_per_km2' in Xf.columns:
        Xf['tree_x_served'] = Xf['tree_density_per_km2']*Xf['served']
    Xf = sm.add_constant(Xf)
    m_int = sm.OLS(sub_full['temp_mean'],Xf).fit(cov_type='HC3')
    print(f'\nInteraction model  R²={m_int.rsquared:.4f}  n={len(sub_full)}')
    print(m_int.summary2().tables[1].to_string())

# Green desert analysis
if 'ndvi_mean' in df.columns and 'tree_density_per_km2' in df.columns:
    ndvi_q = df['ndvi_mean'].quantile(0.33)
    tree_q = df['tree_density_per_km2'].quantile(0.33)
    green_desert = df[(df['ndvi_mean']<ndvi_q)&(df['tree_density_per_km2']<tree_q)]
    if 'served' in df.columns:
        green_desert = green_desert[green_desert['served']==0]
    print(f'\nGreen desert buurten (low NDVI + low tree density + unserved): {len(green_desert)}')
    if len(green_desert)>0:
        print(green_desert[['buurtnaam','ndvi_mean','tree_density_per_km2','temp_mean']].head(10).to_string(index=False))

---
## Section 20: RQ6 — Shade-Weighted Routing (conditional on OSMnx)

In [ ]:
if not (HAS_OSMNX and HAS_GEO and G_walk is not None):
    print('OSMnx or geopandas not available — shade-weighted routing skipped.')
    print('Install: pip install osmnx geopandas')
else:
    # Load shade geojsons
    shade_files = list(SHADE_DIR.glob('*.geojson')) if SHADE_DIR.exists() else []
    if not shade_files:
        print(f'No shade geojsons found in {SHADE_DIR} — skipping routing.')
    else:
        print(f'Loading {len(shade_files)} shade files...')
        shade_gdfs = [gpd.read_file(f) for f in shade_files]
        shade_gdf  = gpd.pd.concat(shade_gdfs, ignore_index=True)
        if shade_gdf.crs is None: shade_gdf = shade_gdf.set_crs('EPSG:4326')
        shade_gdf  = shade_gdf.to_crs('EPSG:28992')
        print(f'Shade segments: {len(shade_gdf)}')

        # Get top-10 HVI buurten
        if 'hvi' in df.columns and gdf is not None and KOEL_CSV.exists():
            koel = pd.read_csv(KOEL_CSV).dropna(subset=['latitude','longitude'])
            top10_hvi_idx = df['hvi'].nlargest(10).index
            route_results = []
            for idx in top10_hvi_idx:
                row = df.loc[idx]
                if pd.isna(row.get('_clat')): continue
                try:
                    orig = ox.nearest_nodes(G_walk, row['_clon'], row['_clat'])
                    best_path=None; best_len=np.inf
                    for _, kr in koel.iterrows():
                        dest = ox.nearest_nodes(G_walk, kr['longitude'], kr['latitude'])
                        try:
                            path = nx.shortest_path(G_walk, orig, dest, weight='length')
                            l = nx.shortest_path_length(G_walk, orig, dest, weight='length')
                            if l<best_len: best_len=l; best_path=path
                        except nx.NetworkXNoPath: pass
                    route_results.append({'buurtnaam':row['buurtnaam'],'dist_km':best_len/1000})
                    print(f'  {row["buurtnaam"]}: {best_len/1000:.2f} km to nearest koelteplek')
                except Exception as e:
                    print(f'  {row["buurtnaam"]}: routing error {e}')
            if route_results:
                rdf = pd.DataFrame(route_results)
                if 'svi_pca' in df.columns:
                    rdf = rdf.merge(df[['buurtnaam','svi_pca']], on='buurtnaam', how='left')
                    rs,ps = stats.spearmanr(rdf['svi_pca'].dropna(), rdf['dist_km'].dropna())
                    print(f'\nEquity test: Spearman ρ(SVI, route_distance) = {rs:.3f} p={ps:.4f}')
                    print(f'{"Vulnerable buurten have longer routes (inequitable)" if rs>0.2 and ps<0.05 else "No significant inequity in route distances"}')

---
## Section 21: Bootstrap & Sensitivity Analysis

In [ ]:
np.random.seed(42)
N_BOOT = 1000

# Weight perturbation: vary HVI weights ±0.10
print('Weight sensitivity analysis...')
w_heat_range = [0.30,0.35,0.40,0.45,0.50]
w_svi_range  = [0.30,0.35,0.40,0.45,0.50]
base_rank = df['hvi'].rank()
stab_results = []
for wh in w_heat_range:
    for ws in w_svi_range:
        wc = max(0.0, 1.0-wh-ws)
        if HAS_GEO and 'svi_pca' in df.columns and 'cooling_access' in df.columns:
            alt = wh*df['hi_norm'] + ws*df['svi_pca'].fillna(0.5) + wc*(1-df['cooling_access'].fillna(0.5))
            rho,_ = stats.spearmanr(base_rank.dropna(), alt.dropna())
            stab_results.append({'w_heat':wh,'w_svi':ws,'w_cool':wc,'spearman_rho':round(rho,4)})
if stab_results:
    stab_df = pd.DataFrame(stab_results)
    print(f'Weight sensitivity — min ρ={stab_df["spearman_rho"].min():.3f}  max={stab_df["spearman_rho"].max():.3f}')
    piv = stab_df.pivot(index='w_heat',columns='w_svi',values='spearman_rho')
    fig,ax=plt.subplots(figsize=(8,6))
    if HAS_SNS: sns.heatmap(piv, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.7, vmax=1.0, ax=ax)
    else: im=ax.imshow(piv.values,cmap='RdYlGn',vmin=0.7,vmax=1.0); plt.colorbar(im,ax=ax)
    ax.set_title('HVI Rank Stability (Spearman ρ vs base)', fontweight='bold')
    ax.set_xlabel('w_svi'); ax.set_ylabel('w_heat')
    plt.tight_layout(); plt.savefig(OUTPUTS/'sensitivity_weights.png',dpi=120,bbox_inches='tight'); plt.show()

In [ ]:
# Bootstrap OLS coefficients for Model A (RQ3)
if 'HI_TOTAAL_S.0' in df.columns and base_x:
    print(f'Bootstrap n={N_BOOT} for RQ3 Model A...')
    sub_b = df[['HI_TOTAAL_S.0']+base_x].dropna().reset_index(drop=True)
    boot_coefs = defaultdict(list)
    for _ in range(N_BOOT):
        samp = sub_b.sample(len(sub_b), replace=True)
        y_b  = samp['HI_TOTAAL_S.0']
        X_b  = sm.add_constant(samp[base_x])
        try:
            m_b = sm.OLS(y_b,X_b).fit()
            for c in base_x:
                if c in m_b.params: boot_coefs[c].append(m_b.params[c])
        except Exception: pass
    print('Bootstrap 95% CI for RQ3 Model A:')
    for c in base_x:
        if boot_coefs[c]:
            lo,hi = np.percentile(boot_coefs[c],[2.5,97.5])
            print(f'  {c[:40]:<42}  [{lo:.4f}, {hi:.4f}]')

# Permutation test: HVI vs HSEI
if 'hvi' in df.columns and 'hsei' in df.columns:
    sub_p = df[['hvi','hsei']].dropna()
    obs_r,_ = stats.pearsonr(sub_p['hvi'],sub_p['hsei'])
    perm_rs = [stats.pearsonr(np.random.permutation(sub_p['hvi'].values),sub_p['hsei'].values)[0]
               for _ in range(N_BOOT)]
    p_perm = np.mean(np.abs(perm_rs)>=abs(obs_r))
    print(f'\nPermutation test HVI vs HSEI: obs r={obs_r:.3f} permutation p={p_perm:.4f}')
    print(f'{"Significant — HVI and HSEI are related" if p_perm<0.05 else "Not significant at α=0.05"}')

# Bootstrap tier boundaries
if 'hvi' in df.columns:
    hvi_vals = df['hvi'].dropna().values
    boot_quantiles = [[np.percentile(np.random.choice(hvi_vals,len(hvi_vals),replace=True),q)
                       for _ in range(N_BOOT)] for q in [20,40,60,80]]
    fig,ax=plt.subplots(figsize=(10,5))
    ax.violinplot(boot_quantiles, positions=[20,40,60,80], showmedians=True)
    ax.set_title('Bootstrap HVI Tier Boundary Distributions', fontweight='bold')
    ax.set_xlabel('HVI percentile (tier boundary)'); ax.set_ylabel('HVI value')
    plt.tight_layout(); plt.savefig(OUTPUTS/'bootstrap_tier_boundaries.png',dpi=120,bbox_inches='tight'); plt.show()

---
## Section 22: Export

In [ ]:
score_cols = [c for c in ['buurtcode','buurtnaam','hvi','hi_norm','svi_pca','cooling_access',
    'hvi_tier','hsei','intervention_priority','priority_class',
    'tree_count','tree_density_per_km2','pct_mature_trees','temp_mean','temp_night',
    'ndvi_mean','water_prc','road_prc','dist_nearest_spot_km',
    'svi_trend_slope','svi_trend_class'] if c in df.columns]

scores_df = df[score_cols].copy()

# CSV exports
scores_df.to_csv(OUTPUTS/'hvi_scores.csv', index=False)
print(f'Exported: {OUTPUTS}/hvi_scores.csv  ({len(scores_df)} rows, {len(score_cols)} cols)')
scores_df.to_csv(CHARLIE/'hvi_scores.csv', index=False)
print(f'Mirrored: {CHARLIE}/hvi_scores.csv')

ip_cols = [c for c in ['buurtcode','buurtnaam','intervention_priority','priority_class',
    'heat_norm','imper_norm','green_norm','tree_norm'] if c in df.columns]
df[ip_cols].to_csv(OUTPUTS/'intervention_priority.csv', index=False)
print(f'Exported: {OUTPUTS}/intervention_priority.csv')

# GeoJSON exports
if HAS_GEO and gdf is not None:
    gdf_out = gdf[['geometry']].copy()
    for c in score_cols:
        if c in df.columns: gdf_out[c] = df[c].values
    gdf_wgs = gdf_out.to_crs('EPSG:4326')

    gdf_wgs.to_file(OUTPUTS/'hvi_map.geojson', driver='GeoJSON')
    print(f'Exported: {OUTPUTS}/hvi_map.geojson')
    gdf_wgs.to_file(CHARLIE/'hvi_map.geojson', driver='GeoJSON')
    print(f'Mirrored: {CHARLIE}/hvi_map.geojson')

    tree_geo_cols = ['geometry'] + [c for c in ['buurtcode','buurtnaam',
        'tree_count','tree_density_per_km2','pct_mature_trees','mean_height_score',
        'tree_species_richness'] if c in df.columns]
    gdf_tree = gdf[['geometry']].copy()
    for c in tree_geo_cols[1:]:
        if c in df.columns: gdf_tree[c] = df[c].values
    gdf_tree.to_crs('EPSG:4326').to_file(OUTPUTS/'tree_canopy.geojson', driver='GeoJSON')
    print(f'Exported: {OUTPUTS}/tree_canopy.geojson')

    ip_cols2 = [c for c in ['buurtcode','buurtnaam','intervention_priority','priority_class'] if c in df.columns]
    gdf_ip = gdf[['geometry']].copy()
    for c in ip_cols2:
        if c in df.columns: gdf_ip[c] = df[c].values
    gdf_ip.to_crs('EPSG:4326').to_file(OUTPUTS/'intervention_priority.geojson', driver='GeoJSON')
    print(f'Exported: {OUTPUTS}/intervention_priority.geojson')

    if 'svi_trend_slope' in df.columns and df['svi_trend_slope'].notna().sum()>0:
        gdf_vt = gdf[['geometry']].copy()
        for c in ['buurtcode','buurtnaam','svi_trend_slope','svi_trend_class']:
            if c in df.columns: gdf_vt[c] = df[c].values
        gdf_vt.to_crs('EPSG:4326').to_file(OUTPUTS/'vuln_trend.geojson', driver='GeoJSON')
        print(f'Exported: {OUTPUTS}/vuln_trend.geojson')
else:
    print('No geometry — skipping GeoJSON exports. Only CSV files were exported.')

print('\n=== EXPORT SUMMARY ===')
for f in sorted(OUTPUTS.glob('*')):
    sz = f.stat().st_size
    print(f'  {f.name:<40} {sz/1024:.1f} KB')